# RAG Document Assistant — Learning Notebook

This notebook teaches you every concept step by step.
Run each cell in order, read the explanations.

---

## Step 0 — Setup

In [ ]:
# Install dependencies (run this cell once)
# !pip install -r ../requirements.txt

import os, sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')
print('Setup complete ✅')

## Step 1 — Embeddings (the core concept of RAG)

An **embedding** turns text into a list of numbers (a vector).
Similar sentences get similar vectors.

Example:
- 'The cat sat on the mat'  → [0.12, -0.34, 0.78, ...]
- 'A feline rested on a rug' → [0.11, -0.33, 0.76, ...]   ← very similar!
- 'The stock market crashed'  → [-0.45, 0.91, -0.23, ...]  ← very different

RAG uses this to find which chunks of your document are most relevant to a question.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

# Load the embedding model (downloads ~90MB once)
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    encode_kwargs={'normalize_embeddings': True},
)

# Embed some example sentences
sentences = [
    'The refund policy allows returns within 30 days.',
    'Customers can return products in 30 days for a full refund.',
    'The CEO announced record profits last quarter.',
]

vecs = embeddings.embed_documents(sentences)
print(f'Each vector has {len(vecs[0])} dimensions')

# Cosine similarity: 1.0 = identical, 0.0 = unrelated
def cosine_sim(a, b):
    return np.dot(a, b)  # vectors are already normalized

print(f'\nSimilarity: sentence 0 vs 1 (same meaning): {cosine_sim(vecs[0], vecs[1]):.3f}')
print(f'Similarity: sentence 0 vs 2 (different topic): {cosine_sim(vecs[0], vecs[2]):.3f}')

## Step 2 — Chunking

We split documents into small chunks before embedding.

**Why?** LLMs have a context window limit (~8000 tokens for LLaMA 3).
We can't embed a 100-page PDF as one vector — we need to split it.

**Overlap** means chunk N ends with the same text that chunk N+1 starts with.
This prevents losing context at chunk boundaries.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

sample_text = """
Artificial intelligence (AI) is intelligence demonstrated by machines.
Unlike natural intelligence displayed by animals and humans, AI is a
branch of computer science that builds smart machines.

Machine learning is a subset of AI. It gives systems the ability to
automatically learn and improve from experience without being explicitly
programmed.

Deep learning is part of a broader family of machine learning methods.
It uses neural networks with many layers to model and understand data.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,    # small so we can see it clearly
    chunk_overlap=30,  # overlap between chunks
)

chunks = splitter.split_text(sample_text)
print(f'Split into {len(chunks)} chunks:\n')
for i, chunk in enumerate(chunks):
    print(f'--- Chunk {i+1} ({len(chunk)} chars) ---')
    print(chunk)
    print()

## Step 3 — Vector Store (ChromaDB)

ChromaDB stores your chunks + their embeddings on disk.
When you ask a question, it finds the most similar chunks in milliseconds
using an algorithm called ANN (Approximate Nearest Neighbor search).

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain.schema import Document

# Create some test documents
docs = [
    Document(page_content='The refund policy allows returns within 30 days of purchase.',
             metadata={'source': 'policy.pdf', 'page': 1}),
    Document(page_content='Our products come with a 1-year warranty.',
             metadata={'source': 'policy.pdf', 'page': 2}),
    Document(page_content='The CEO earned $5 million in salary last year.',
             metadata={'source': 'annual_report.pdf', 'page': 4}),
    Document(page_content='Revenue grew 23% year-over-year to $1.2 billion.',
             metadata={'source': 'annual_report.pdf', 'page': 5}),
]

# Store in ChromaDB (in memory for this demo)
vectorstore = Chroma.from_documents(docs, embeddings)

# Search!
query = 'How long do I have to return a product?'
results = vectorstore.similarity_search_with_score(query, k=2)

print(f'Query: "{query}"\n')
for doc, score in results:
    print(f'Score: {score:.3f} | Source: {doc.metadata["source"]}')
    print(f'Content: {doc.page_content}\n')

## Step 4 — Re-ranking (the secret sauce 🔑)

Vector search is fast but imprecise — it finds 'similar' text.
A **cross-encoder** is slower but reads both query and chunk together,
giving a much more accurate relevance score.

Think of it as: vector search is a CTRL+F, re-ranking is a human reading the results.

In [ ]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

query = 'How long do I have to return a product?'
candidate_texts = [
    'The refund policy allows returns within 30 days of purchase.',
    'Our products come with a 1-year warranty.',
    'Revenue grew 23% year-over-year to $1.2 billion.',
]

pairs = [(query, text) for text in candidate_texts]
scores = cross_encoder.predict(pairs)

print('Cross-encoder scores (higher = more relevant):\n')
for text, score in sorted(zip(candidate_texts, scores), key=lambda x: x[1], reverse=True):
    print(f'  Score: {score:6.2f} | {text[:60]}...')

## Step 5 — Full RAG Pipeline End-to-End

Now let's put it all together using our project code.

In [ ]:
# First, ingest a sample document
# (Create a simple test PDF first or use any PDF you have)

# from src.ingestion import ingest_document
# result = ingest_document('../data/uploads/your_document.pdf')
# print(result)

print('Upload a PDF to data/uploads/, then uncomment and run this cell')

In [ ]:
# Query the document
# from src.retrieval import query_documents
# result = query_documents('What is this document about?')
# print('Answer:', result['answer'])
# print('Sources:', result['sources'])

print('After ingesting a document, uncomment and run this cell')

## Step 6 — RAGAS Evaluation

After building the system, you need to MEASURE it.
RAGAS gives objective scores — this is what separates a serious project
from a toy demo in interviews.

In [ ]:
# Run evaluation (requires documents to be ingested)
# from evaluation.evaluate import evaluate_rag
# summary = evaluate_rag()
# print(summary['mean_scores'])

print('After ingesting documents and setting GROQ_API_KEY, run the evaluation')